# 02. Financial Report Generator

Generate a financial summary HTML report for a single bill.
Classifier logic lives in `classify_bill.py`.

In [ ]:
import re
import sys
from datetime import datetime
from html import escape as esc
from pathlib import Path

import pandas as pd

_REPO = next(p for p in [Path().resolve(), *Path().resolve().parents] if (p / "pyproject.toml").exists())
_RESEARCH = _REPO / "docs" / "research" / "financial-semantics"
sys.path.insert(0, str(_RESEARCH))

from classify_bill import build_financial_df, check_coverage, classify_text, group_financial_rows  # noqa: E402

from deltatrack.bill_tree import bill_title, normalize_bill  # noqa: E402

## Load bill

In [ ]:
xml_path = _REPO / "bills" / "118-hr-4366" / "1_reported-in-house.xml"
tree = normalize_bill(xml_path)
df_financial = build_financial_df(tree)
check_coverage(df_financial, tree)
df_financial.head()

## Generate HTML report

In [ ]:
TYPE_COLORS = {
    "appropriation": ("#dcfce7", "#166534"),
    "transfer": ("#cffafe", "#155e75"),
    "rescission": ("#fee2e2", "#991b1b"),
    "restriction": ("#fee2e2", "#991b1b"),
    "authorization": ("#e0f2fe", "#0c4a6e"),
    "cap": ("#fef3c7", "#92400e"),
    "earmark": ("#ede9fe", "#5b21b6"),
    "availability": ("#d1fae5", "#065f46"),
    "sub_allocation": ("#dbeafe", "#1e40af"),
    "directive": ("#e0e7ff", "#3730a3"),
    "fee": ("#ffedd5", "#9a3412"),
    "unknown": ("#f3f4f6", "#374151"),
}

TYPE_DESCRIPTIONS = {
    "appropriation": "Funds directly allocated to an agency or program",
    "transfer": "Funds moved between accounts",
    "rescission": "Previously appropriated funds clawed back",
    "restriction": "Prohibition on how funds may be used",
    "authorization": "Congressional authorization to request an appropriation; funds not yet available for obligation",
    "cap": "Upper limit on spending for a specific purpose",
    "earmark": "Funds designated for a recipient listed in a table",
    "availability": "Time extension specifying when funds may be spent",
    "sub_allocation": "Portion of funds reserved for a specific use",
    "directive": "Mandatory action required of an agency",
    "fee": "Charge imposed on individuals or entities",
    "unknown": "Could not be automatically classified",
}

ALL_TYPES = [
    "appropriation",
    "transfer",
    "rescission",
    "authorization",
    "fee",
    "restriction",
    "directive",
    "cap",
    "earmark",
    "availability",
    "sub_allocation",
    "unknown",
]


def fmt_amount(x):
    return f"${x:,.0f}" if pd.notna(x) else "—"


def type_badge(t):
    bg, fg = TYPE_COLORS.get(t, ("#f3f4f6", "#374151"))
    return (
        f'<span style="background:{bg};color:{fg};padding:2px 7px;'
        f'border-radius:999px;font-size:11px;font-weight:600">{esc(t)}</span>'
    )


def highlight_body(body_text):
    pieces = re.split(r"(\bProvided(?:\s+further)?,?\s+That\b)", body_text, flags=re.IGNORECASE)
    clause_idx = 0
    out = []
    for piece in pieces:
        if re.match(r"\bProvided(?:\s+further)?,?\s+That\b", piece, re.IGNORECASE):
            out.append(f'<strong style="color:#555">{esc(piece)}</strong> ')
        else:
            t = classify_text(body_text) if clause_idx == 0 else classify_text(piece)
            t = t or "unknown"
            bg, fg = TYPE_COLORS.get(t, ("#f3f4f6", "#374151"))
            out.append(
                f'<span style="background:{bg};color:{fg};border-radius:3px;padding:1px 3px">{esc(piece)}</span>'
            )
            clause_idx += 1
    return "".join(out)


def build_legend(df):
    counts = df["type"].value_counts() if not df.empty else {}
    items = []
    for t in ALL_TYPES:
        n = int(counts.get(t, 0))
        count_html = (
            f'<span style="color:#999;font-size:12px;min-width:32px;display:inline-block">({n})</span>'
            if n
            else '<span style="min-width:32px;display:inline-block"></span>'
        )
        items.append(
            '<div style="display:flex;align-items:baseline;gap:8px;padding:3px 0">'
            + type_badge(t)
            + count_html
            + f'<span style="color:#555;font-size:13px">{TYPE_DESCRIPTIONS[t]}</span></div>'
        )
    return (
        '<div style="margin-bottom:20px;border:1px solid #e3ddd7;border-radius:0.625rem;'
        'padding:12px 14px;background:#fff">'
        '<div style="font-weight:600;font-size:14px;margin-bottom:10px">Category key</div>' + "".join(items) + "</div>"
    )


def ordinal(n):
    if 11 <= n % 100 <= 13:
        return f"{n}th"
    return f"{n}{['th', 'st', 'nd', 'rd', 'th'][min(n % 10, 4)]}"


def build_financial_html(df, tree):
    FONT_SANS = "ui-sans-serif, system-ui, -apple-system, 'Segoe UI', Roboto, Arial, sans-serif"
    FONT_MONO = "ui-monospace, 'SF Mono', Menlo, Consolas, monospace"

    css = f"""<style>
* {{ box-sizing: border-box; margin: 0; padding: 0; }}
body {{ background: #f9f7f5; font-family: {FONT_SANS}; color: #1c1c3a;
       font-size: 14px; line-height: 1.6; margin: 24px; }}
h2 {{ font-family: ui-serif, Georgia, serif; font-size: 20px; margin-bottom: 12px;
     letter-spacing: -0.02em; }}
table {{ width: 100%; border-collapse: collapse; background: #fff;
         border-radius: 0.625rem; table-layout: fixed;
         box-shadow: 0 1px 2px rgba(28,28,58,.04), 0 1px 3px rgba(28,28,58,.06);
         overflow: hidden; }}
th {{ background: #eeeae6; text-align: left; padding: 9px 12px;
      border-bottom: 2px solid #e3ddd7; font-weight: 600; font-size: 13px; }}
td {{ padding: 8px 12px; border-bottom: 1px solid #e3ddd7; vertical-align: middle; }}
.amount {{ text-align: right; font-family: {FONT_MONO};
           font-variant-numeric: tabular-nums; white-space: nowrap; }}
.row-primary {{ background: #f0ede9; font-weight: 600; cursor: pointer; user-select: none; }}
.row-primary:hover {{ filter: brightness(0.97); }}
.row-primary td:first-child::before {{ content: '\\25b6'; font-size: 10px; margin-right: 8px;
                                       color: #888; display: inline-block;
                                       transition: transform 0.15s; }}
.row-primary.open td:first-child::before {{ transform: rotate(90deg); }}
.detail-inner {{ display: flex; }}
.detail-amounts {{ min-width: 220px; max-width: 260px; padding: 12px;
                   border-right: 1px solid #e3ddd7; display: flex;
                   flex-direction: column; gap: 10px; }}
.detail-amount-item {{ display: flex; flex-direction: column; gap: 3px; }}
.detail-amount-value {{ font-family: {FONT_MONO}; font-size: 15px;
                        font-variant-numeric: tabular-nums; }}
.detail-text {{ padding: 12px 16px; font-size: 13px; line-height: 1.9;
                white-space: pre-wrap; flex: 1; }}
.badge-review {{ background: #fef3c7; color: #92400e; font-size: 11px;
                 padding: 1px 6px; border-radius: 999px; margin-left: 6px; font-weight: 500; }}
.count-badge {{ font-size: 11px; color: #888; font-weight: 400; margin-left: 6px; }}
.sort-bar {{ display: flex; gap: 8px; align-items: center; margin-bottom: 12px; }}
.sort-bar span {{ font-size: 13px; color: #888; margin-right: 2px; }}
.sort-btn {{ padding: 4px 12px; border: 1px solid #e3ddd7; border-radius: 999px;
             background: #fff; font-size: 13px; cursor: pointer;
             font-family: inherit; color: #1c1c3a; }}
.sort-btn:hover {{ background: #eeeae6; }}
.sort-btn.active {{ background: #1c1c3a; color: #fff; border-color: #1c1c3a; }}
</style>"""

    js = """<script>
function toggleDetail(el) {
    var id = el.dataset.group;
    var open = el.classList.toggle('open');
    document.getElementById('detail-' + id).style.display = open ? '' : 'none';
    var state = JSON.parse(localStorage.getItem('fin-open') || '{}');
    state[id] = open;
    localStorage.setItem('fin-open', JSON.stringify(state));
}

function sortTable(key, btn) {
    var tbody = document.querySelector('#fin-table tbody');
    var rows = Array.from(tbody.querySelectorAll('tr'));
    var groups = [];
    var i = 0;
    while (i < rows.length) {
        if (rows[i].classList.contains('row-primary')) {
            groups.push([rows[i], rows[i + 1]]);
            i += 2;
        } else { i++; }
    }
    if (key === 'amount') {
        groups.sort(function(a, b) {
            return parseFloat(b[0].dataset.amount || 0) - parseFloat(a[0].dataset.amount || 0);
        });
    } else if (key === 'review') {
        groups.sort(function(a, b) {
            return (b[0].dataset.review === '1' ? 1 : 0) - (a[0].dataset.review === '1' ? 1 : 0);
        });
    } else {
        groups.sort(function(a, b) {
            return parseInt(a[0].dataset.group) - parseInt(b[0].dataset.group);
        });
    }
    var frag = document.createDocumentFragment();
    groups.forEach(function(g) { frag.appendChild(g[0]); frag.appendChild(g[1]); });
    tbody.appendChild(frag);
    document.querySelectorAll('.sort-btn').forEach(function(b) { b.classList.remove('active'); });
    btn.classList.add('active');
}

window.addEventListener('DOMContentLoaded', function() {
    var state = JSON.parse(localStorage.getItem('fin-open') || '{}');
    Object.keys(state).forEach(function(id) {
        if (state[id]) {
            var row = document.querySelector('[data-group="' + id + '"]');
            if (row) {
                row.classList.add('open');
                document.getElementById('detail-' + id).style.display = '';
            }
        }
    });
});
</script>"""

    groups = group_financial_rows(df)

    rows_html = []
    for gid, group in enumerate(groups):
        primary = group[0]
        subs = group[1:]
        needs_review = any(r["needs_review"] for r in group)
        review_badge = '<span class="badge-review">⚠ review</span>' if needs_review else ""
        count_badge = f'<span class="count-badge">({len(subs)})</span>' if subs else ""
        amount_val = primary["amount"] if pd.notna(primary["amount"]) else 0

        rows_html.append(
            f'<tr class="row-primary" onclick="toggleDetail(this)" data-group="{gid}"'
            f' data-amount="{amount_val}" data-review="{"1" if needs_review else "0"}">'
            f"<td>{esc(primary['department'])}</td><td>{esc(primary['account'])}{count_badge}{review_badge}</td>"
            f"<td>{type_badge(primary['type'])}</td>"
            f'<td class="amount">{fmt_amount(primary["amount"])}</td>'
            f"</tr>"
        )

        amount_items = "".join(
            f'<div class="detail-amount-item">'
            f"{type_badge(r['type'])}"
            f'<span class="detail-amount-value">{fmt_amount(r["amount"])}</span>'
            f"</div>"
            for r in group
        )
        highlighted = highlight_body(primary["body_text"]) if primary["body_text"] else ""

        rows_html.append(
            f'<tr id="detail-{gid}" style="display:none">'
            f'<td colspan="4" style="padding:0">'
            f'<div class="detail-inner">'
            f'<div class="detail-amounts">{amount_items}</div>'
            f'<div class="detail-text">{highlighted}</div>'
            f"</div></td></tr>"
        )

    sort_bar = (
        '<div class="sort-bar"><span>Sort:</span>'
        '<button class="sort-btn active" onclick="sortTable(\'default\', this)">Default</button>'
        '<button class="sort-btn" onclick="sortTable(\'amount\', this)">Amount ↓</button>'
        '<button class="sort-btn" onclick="sortTable(\'review\', this)">Needs review first</button>'
        "</div>"
    )
    bill_header = (
        f'<div style="margin-bottom:20px">'
        f'<h1 style="font-family:ui-serif,Georgia,serif;font-size:24px;'
        f'letter-spacing:-0.02em;margin-bottom:4px">{esc(bill_title(tree))}</h1>'
        f'<p style="color:#888;font-size:14px">{ordinal(tree.congress)} Congress</p>'
        f"</div>"
    )
    return (
        css
        + js
        + bill_header
        + build_legend(df)
        + "<h2>Financial Clauses</h2>"
        + sort_bar
        + '<table id="fin-table"><thead><tr>'
        + '<th style="width:35%">Department</th><th style="width:40%">Account</th>'
        + '<th style="width:12%">Type</th>'
        + '<th style="width:13%">Amount</th>'
        + "</tr></thead>"
        + "<tbody>"
        + "".join(rows_html)
        + "</tbody></table>"
    )


ts = datetime.now().strftime("%H:%M:%S")

html = (
    f"<html><head>"
    f"<title>{esc(bill_title(tree))}</title>"
    f"</head><body>"
    f'<p style="color:#999;font-size:12px;margin-bottom:16px">Last updated: {ts}</p>'
    + build_financial_html(df_financial, tree)
    + "</body></html>"
)

fname = f"financial_{tree.congress}_{tree.bill_type}_{tree.bill_number}.html"
with open(fname, "w", encoding="utf-8") as f:
    f.write(html)

print(f"Saved to {fname}")

In [ ]:
from classify_bill import CAP_AMOUNT_RE, DOLLAR, split_clauses  # noqa: E402

# Tag values assigned by GPO XML encoding; matches bill_tree.py node.tag field.
# If bill_tree.py renames these tags this set must be updated to match.
APPRO_TAGS = {"appropriations-intermediate", "appropriations-small", "appropriations-major"}

summary_rows = []
for n in tree.nodes:
    if not DOLLAR.search(n.body_text or ""):
        continue
    node_label = classify_text(n.body_text)
    for clause_text, level in split_clauses(n.body_text):
        if not DOLLAR.search(clause_text):
            continue
        label = node_label if level == "primary" else classify_text(clause_text)
        if label != "appropriation":
            continue
        m = DOLLAR.search(CAP_AMOUNT_RE.sub("", clause_text)) or DOLLAR.search(clause_text)
        if not m:
            continue
        amount = float(m.group(1).replace(",", ""))
        department = n.display_path[0] if n.display_path else ""
        account = n.display_path[-1] if n.display_path else ""
        summary_rows.append({"department": department, "account": account, "amount": amount})

df_summary = pd.DataFrame(summary_rows) if summary_rows else pd.DataFrame(columns=["department", "account", "amount"])
# sort=False preserves bill order (i.e. title/division order), not alphabetical.
dept_account = df_summary.groupby(["department", "account"], sort=False)["amount"].sum().reset_index()
dept_totals = df_summary.groupby("department", sort=False)["amount"].sum().reset_index()


def build_summary_html(dept_totals, dept_account):
    FONT_MONO = "ui-monospace, 'SF Mono', Menlo, Consolas, monospace"

    js = (
        "<script>\n"
        "function toggleDept(el) {\n"
        "    var dept = el.dataset.dept;\n"
        "    var open = el.classList.toggle('dept-open');\n"
        "    document.querySelectorAll('.acct-' + dept).forEach(function(r) {\n"
        "        r.style.display = open ? '' : 'none';\n"
        "    });\n"
        "    el.querySelector('.dept-arrow').style.transform = open ? 'rotate(90deg)' : '';\n"
        "}\n"
        "</script>"
    )

    css = (
        "<style>\n"
        ".summary-wrap { margin-bottom: 24px; }\n"
        ".summary-wrap > summary { font-family: ui-serif, Georgia, serif; font-size: 20px;\n"
        "    font-weight: 600; letter-spacing: -0.02em; cursor: pointer; color: #1c1c3a;\n"
        "    list-style: none; padding: 0; margin-bottom: 12px; }\n"
        ".summary-wrap > summary::before { content: '\\25b6'; font-size: 10px;\n"
        "    margin-right: 8px; color: #888; display: inline-block; transition: transform 0.15s; }\n"
        "details[open].summary-wrap > summary::before { transform: rotate(90deg); }\n"
        f"#summary-table {{ width: 100%; border-collapse: collapse; margin-top: 12px; table-layout: fixed; }}\n"
        f"#summary-table th {{ background: #eeeae6; text-align: left; padding: 9px 12px;\n"
        f"    border-bottom: 2px solid #e3ddd7; font-weight: 600; font-size: 13px; }}\n"
        f"#summary-table td {{ padding: 8px 12px; border-bottom: 1px solid #e3ddd7; vertical-align: middle; }}\n"
        ".dept-row { background: #f0ede9; font-weight: 600; cursor: pointer; user-select: none; }\n"
        ".dept-row:hover { filter: brightness(0.97); }\n"
        ".dept-arrow { display: inline-block; font-size: 10px; margin-right: 8px;\n"
        "    color: #888; transition: transform 0.15s; }\n"
        ".acct-row td:first-child { padding-left: 28px; }\n"
        f".sum-amount {{ text-align: right; font-family: {FONT_MONO};\n"
        "    font-variant-numeric: tabular-nums; white-space: nowrap; }}\n"
        ".summary-note { color: #888; font-size: 12px; margin-top: 8px; }\n"
        "</style>"
    )

    rows_html = []
    for i, dept_row in enumerate(dept_totals.itertuples()):
        dept = dept_row.department
        dept_id = str(i)
        accounts = dept_account[dept_account["department"] == dept]
        rows_html.append(
            f'<tr class="dept-row" onclick="toggleDept(this)" data-dept="{dept_id}">'
            f'<td><span class="dept-arrow">&#9658;</span>{esc(dept)}</td>'
            f'<td class="sum-amount">${dept_row.amount:,.0f}</td>'
            f"</tr>"
        )
        for acct_row in accounts.itertuples():
            rows_html.append(
                f'<tr class="acct-row acct-{dept_id}" style="display:none">'
                f"<td>{esc(acct_row.account)}</td>"
                f'<td class="sum-amount">${acct_row.amount:,.0f}</td>'
                f"</tr>"
            )

    table = (
        '<table id="summary-table"><thead><tr>'
        '<th style="width:80%">Department / Account</th>'
        '<th style="width:20%">Appropriations</th>'
        "</tr></thead><tbody>" + "".join(rows_html) + "</tbody></table>"
    )
    note = '<p class="summary-note">Totals include direct appropriation amounts only.</p>'
    return (
        js
        + css
        + '<details class="summary-wrap" open>'
        + "<summary>Appropriations Summary</summary>"
        + table
        + note
        + "</details>"
    )


summary_html = build_summary_html(dept_totals, dept_account)
# fname and tree are defined in the cell above; ts re-read here so the
# "Last updated" timestamp reflects the full render including the summary.
ts = datetime.now().strftime("%H:%M:%S")
_fin_html = build_financial_html(df_financial, tree)
# Inject summary panel immediately before the "Financial Clauses" heading.
_fin_html = _fin_html.replace(
    "<h2>Financial Clauses</h2>",
    summary_html + "<h2>Financial Clauses</h2>",
    1,
)
html = (
    f"<html><head><title>{esc(bill_title(tree))}</title></head><body>"
    f'<p style="color:#999;font-size:12px;margin-bottom:16px">Last updated: {ts}</p>' + _fin_html + "</body></html>"
)
with open(fname, "w", encoding="utf-8") as f:
    f.write(html)
print(f"Updated {fname} with agency summary ({len(dept_totals)} departments, {len(dept_account)} accounts)")